# Transfer-Performance Matching
This notebook focuses on creating performance block/age/season/transfer value pairs,
which can ultimately be used to train and test a supervised Bayesian linear regression
model to predict transfer values.

In [1]:
import pandas as pd
import duckdb
from understatapi import UnderstatClient
import numpy as np
import os
import sys

# Also import local helpers
target_dir = os.path.abspath('..') 

if target_dir not in sys.path:
    sys.path.append(target_dir)
    
from preprocess import merge_stats_df_with_transfermarkt

In [ ]:
# 1. Get position data
positions = ['M']
stats_df = pd.read_csv(f"../data/{'_'.join(positions)}_stats_2.csv")

stats = ["xG", "xA", "key_passes", "xGChain", "xGBuildup"]
per_90_stats = list(map(lambda s: f"{s}_per_90", stats))

# Filter stat columns to what we want
cols = list(filter(lambda x: x[-6:] != "per_90" or x in per_90_stats, stats_df.columns))
stats_df = stats_df[cols]

# Train/test split by player
all_players = stats_df['player_id'].unique()

# 80/20 split
np.random.seed(42)
train_players = np.random.choice(all_players, size=int(0.8*len(all_players)), replace=False)

train_df = stats_df[stats_df['player_id'].isin(train_players)]
test_df = stats_df[~stats_df['player_id'].isin(train_players)]

train_df.head()

,player_id,player_name,date,xG_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,league
0,150,Josuha Guilavogui,2014-11-22 14:30:00,0.040743,0.035318,0.690537,0.250601,0.211696,Bundesliga
1,150,Josuha Guilavogui,2015-03-22 14:30:00,0.021087,0.005730,0.284810,0.287882,0.270286,Bundesliga
2,150,Josuha Guilavogui,2015-09-12 17:30:00,0.079120,0.025244,0.855784,0.487742,0.397243,Bundesliga
3,150,Josuha Guilavogui,2015-11-29 20:30:00,0.072270,0.010650,0.613915,0.398019,0.378410,Bundesliga
4,150,Josuha Guilavogui,2016-03-19 18:30:00,0.047304,0.031790,0.793451,0.377149,0.342835,Bundesliga


In [3]:
train_df.head()

,player_id,player_name,date,xG_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,league
0,150,Josuha Guilavogui,2014-11-22 14:30:00,0.040743,0.035318,0.690537,0.250601,0.211696,Bundesliga
1,150,Josuha Guilavogui,2015-03-22 14:30:00,0.021087,0.005730,0.284810,0.287882,0.270286,Bundesliga
2,150,Josuha Guilavogui,2015-09-12 17:30:00,0.079120,0.025244,0.855784,0.487742,0.397243,Bundesliga
3,150,Josuha Guilavogui,2015-11-29 20:30:00,0.072270,0.010650,0.613915,0.398019,0.378410,Bundesliga
4,150,Josuha Guilavogui,2016-03-19 18:30:00,0.047304,0.031790,0.793451,0.377149,0.342835,Bundesliga


In [4]:
# 2. Pull transfer value data and merge
train_df_combined = merge_stats_df_with_transfermarkt(train_df, use_transfermarkt_info=True)
test_df_combined = merge_stats_df_with_transfermarkt(test_df, use_transfermarkt_info=True)

/Users/lancehendricks/Documents/College Coding/ML 2/TransferValues/src/preprocess/player_stats.py:271: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stats_df["date"] = stats_df['date'].astype('datetime64[us]')
/Users/lancehendricks/Documents/College Coding/ML 2/TransferValues/src/preprocess/player_stats.py:271: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stats_df["date"] = stats_df['date'].astype('datetime64[us]')


In [5]:
train_df_combined.head()

,player_id,player_name,date,date_of_birth,league,xG_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,value
7541,78.0,Tarik Elyounoussi,2014-11-03,1988-02-23,Bundesliga,0.391003,0.112040,1.348315,0.551466,0.101346,4500000
7583,2254.0,Mateo Kovacic,2014-11-08,1994-05-06,Serie_A,0.087236,0.172524,2.601156,0.372742,0.226015,22000000
7584,1291.0,Hernanes,2014-11-08,1985-05-29,Serie_A,0.137678,0.278141,3.360390,0.641700,0.435994,18000000
7597,1229.0,Luis Muriel,2014-11-08,1991-04-16,Serie_A,0.235126,0.372327,2.793103,0.699868,0.321844,11000000
8149,3563.0,Florent Balmont,2014-11-20,1980-02-02,Ligue_1,0.026185,0.256069,2.324247,0.258315,0.125754,750000


In [6]:
# Add age and year (fractional columns
train_df_combined["age"] = (train_df_combined["date"] - train_df_combined["date_of_birth"]).dt.days  / 365.25 # .25 accounts for leap years
test_df_combined["age"] = (test_df_combined["date"] - test_df_combined["date_of_birth"]).dt.days  / 365.25


train_df_combined["year"] = train_df_combined["date"].dt.year + (train_df_combined["date"].dt.day_of_year / 365.25)
test_df_combined["year"] = test_df_combined["date"].dt.year + (test_df_combined["date"].dt.day_of_year / 365.25)

train_df_combined.head()

,player_id,player_name,date,date_of_birth,league,xG_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,value,age,year
7541,78.0,Tarik Elyounoussi,2014-11-03,1988-02-23,Bundesliga,0.391003,0.112040,1.348315,0.551466,0.101346,4500000,26.694045,2014.840520
7583,2254.0,Mateo Kovacic,2014-11-08,1994-05-06,Serie_A,0.087236,0.172524,2.601156,0.372742,0.226015,22000000,20.509240,2014.854209
7584,1291.0,Hernanes,2014-11-08,1985-05-29,Serie_A,0.137678,0.278141,3.360390,0.641700,0.435994,18000000,29.445585,2014.854209
7597,1229.0,Luis Muriel,2014-11-08,1991-04-16,Serie_A,0.235126,0.372327,2.793103,0.699868,0.321844,11000000,23.564682,2014.854209
8149,3563.0,Florent Balmont,2014-11-20,1980-02-02,Ligue_1,0.026185,0.256069,2.324247,0.258315,0.125754,750000,34.798084,2014.887064


In [7]:
# Get rid of DOB column
train_df_combined = train_df_combined.drop("date_of_birth", axis=1)
test_df_combined = test_df_combined.drop("date_of_birth", axis=1)

In [8]:
# Save to csv
train_df_combined.to_csv(f"../data/{'_'.join(positions)}_stats_values_train.csv", index=False)
test_df_combined.to_csv(f"../data/{'_'.join(positions)}_stats_values_test.csv", index=False)